## DDL Gold: pf.gold.dim_organizacion  (SCD Type 1)
## Los atributos se sobrescriben (no se guarda historial).

In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.dim_organizacion;

CREATE TABLE IF NOT EXISTS pf.gold.dim_organizacion (
    org_sk BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    org_id STRING NOT NULL COMMENT 'BK - organizacion natural',
    org_nombre STRING COMMENT 'Nombre legible (referencial)',
    num_modelos_ref BIGINT COMMENT 'Nº modelos de referencia (recalculable)',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (org_sk),
    CONSTRAINT uniq_dim_organizacion UNIQUE (org_id)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Organizacion - SCD Type 1';

In [0]:
%sql

-- Carga/actualizacion SCD1: si el org_id ya existe, se sobrescriben atributos.
MERGE INTO pf.gold.dim_organizacion AS t
USING (
    SELECT org_id AS org_id,
           org_id AS org_nombre,
           COUNT(DISTINCT model_id) AS num_modelos_ref
    FROM pf.silver.modelos
    GROUP BY org_id
) AS s
ON t.org_id = s.org_id
WHEN MATCHED THEN
    UPDATE SET t.org_nombre = s.org_nombre,
               t.num_modelos_ref = s.num_modelos_ref
WHEN NOT MATCHED THEN
    INSERT (org_id, org_nombre, num_modelos_ref, _createdAt)
    VALUES (s.org_id, s.org_nombre, s.num_modelos_ref, CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT 
    org_sk, 
    org_id, 
    org_nombre, 
    num_modelos_ref 
FROM pf.gold.dim_organizacion 
ORDER BY num_modelos_ref DESC 
LIMIT 10;